# F3_04_Nucleo_POO

    ## Objetivo

    Implementar el núcleo algorítmico orientado a objetos del sistema, evidenciando encapsulamiento, herencia, polimorfismo, clases cohesivas, bajo acoplamiento y métodos documentados.

    ## Resultado esperado

    - Clase `DatasetFitness`.
    - Clase base `AlgoritmoBase`.
    - Subclases `RankingFitness`, `MaximoFitness`, `PromedioRegionalFitness`.
    - Clase `MedidorComplejidad`.
    - Clase `ExperimentoAlgoritmico`.
    - Ejecución polimórfica con método `ejecutar()`.
    - Mediciones de algoritmos POO.

> Nota técnica: este notebook está diseñado para ejecutarse desde cualquier subcarpeta del repositorio. 
> Las rutas se resuelven buscando automáticamente la carpeta raíz `gym-fitness-analytics`.

In [1]:
from pathlib import Path
import time
import tracemalloc

import pandas as pd
import numpy as np

## 1. Carga y preparación del dataset

In [2]:
def encontrar_raiz_proyecto(nombre_repo="gym-fitness-analytics"):
    ruta_actual = Path.cwd().resolve()

    for ruta in [ruta_actual] + list(ruta_actual.parents):
        if ruta.name == nombre_repo:
            return ruta

    raise FileNotFoundError(
        f"No se pudo encontrar la raíz del proyecto '{nombre_repo}' desde {ruta_actual}"
    )


PROJECT_ROOT = encontrar_raiz_proyecto()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "gym_data_processed.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset procesado: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)

print("Dataset cargado correctamente.")
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

columnas_f3 = [
    "country",
    "region",
    "year",
    "gym_penetration_rate",
    "memberships_per_100k",
    "gyms_per_100k",
    "revenue_per_membership_usd",
    "periodo"
]

faltantes = [col for col in columnas_f3 if col not in df.columns]

if faltantes:
    raise ValueError(f"Faltan columnas necesarias: {faltantes}")

df_f3 = df[columnas_f3].dropna(
    subset=[
        "gym_penetration_rate",
        "memberships_per_100k",
        "gyms_per_100k"
    ]
).copy()

print("Dataset preparado para núcleo POO.")
print("Filas:", df_f3.shape[0])

df_f3.head()

Dataset cargado correctamente.
Filas: 3564
Columnas: 32
Dataset preparado para núcleo POO.
Filas: 3564


,country,region,year,gym_penetration_rate,memberships_per_100k,gyms_per_100k,revenue_per_membership_usd,periodo
0,Angola,Africa,2000,0.0059,589.822616,1.259658,59.999990,pre_covid
1,Angola,Africa,2001,0.0062,620.043651,1.325594,59.999730,pre_covid
2,Angola,Africa,2002,0.0070,698.840625,1.437006,60.000025,pre_covid
3,Angola,Africa,2003,0.0080,795.727216,1.566008,59.999902,pre_covid
4,Angola,Africa,2004,0.0097,965.650082,1.747272,60.000100,pre_covid


## 2. Clase `DatasetFitness`

Encapsula el DataFrame y controla acceso/validación.

In [3]:
class DatasetFitness:
    """
    Clase responsable de encapsular el DataFrame del proyecto fitness.

    Principios aplicados:
        - Encapsulamiento: el DataFrame se almacena como atributo protegido.
        - Alta cohesión: la clase gestiona validaciones y acceso a datos.
        - Bajo acoplamiento: otras clases reciben esta clase o datos ya preparados.
    """

    def __init__(self, dataframe):
        if not isinstance(dataframe, pd.DataFrame):
            raise TypeError("El parámetro dataframe debe ser un pd.DataFrame.")

        self._df = dataframe.copy()

    @property
    def df(self):
        """
        Retorna una copia del DataFrame para evitar modificaciones externas directas.
        """
        return self._df.copy()

    def validar_columnas(self, columnas):
        faltantes = [col for col in columnas if col not in self._df.columns]

        if faltantes:
            raise ValueError(f"Faltan columnas requeridas: {faltantes}")

        return True

    def obtener_columna_numerica(self, columna):
        self.validar_columnas([columna])

        return (
            self._df[columna]
            .dropna()
            .astype(float)
            .tolist()
        )

    def filtrar_por_periodo(self, periodo):
        self.validar_columnas(["periodo"])

        periodos_validos = {"pre_covid", "covid", "post_covid"}

        if periodo not in periodos_validos:
            raise ValueError(
                f"Periodo inválido: {periodo}. Use uno de: {periodos_validos}"
            )

        return self._df[self._df["periodo"] == periodo].copy()

    def resumen(self):
        resumen = {
            "filas": self._df.shape[0],
            "columnas": self._df.shape[1]
        }

        if "country" in self._df.columns:
            resumen["paises"] = self._df["country"].nunique()

        if "region" in self._df.columns:
            resumen["regiones"] = self._df["region"].nunique()

        if "year" in self._df.columns:
            resumen["anio_min"] = int(self._df["year"].min())
            resumen["anio_max"] = int(self._df["year"].max())

        return resumen

## 3. Clase base `AlgoritmoBase`

Permite herencia y polimorfismo. Las subclases implementan `ejecutar()`.

In [4]:
class AlgoritmoBase:
    """
    Clase base para algoritmos del proyecto.

    Esta clase permite demostrar herencia y polimorfismo.
    Las subclases deben implementar el método ejecutar().
    """

    def __init__(self, nombre):
        self._nombre = nombre

    @property
    def nombre(self):
        return self._nombre

    def ejecutar(self):
        raise NotImplementedError(
            "El método ejecutar() debe ser implementado por las subclases."
        )

## 4. Subclase `RankingFitness`

In [5]:
class RankingFitness(AlgoritmoBase):
    """
    Algoritmo de ranking de países/registros según una variable numérica.
    """

    def __init__(self, dataframe, columna, top_n=10):
        super().__init__("Ranking Fitness")

        self._df = dataframe.copy()
        self._columna = columna
        self._top_n = top_n

    def ejecutar(self):
        if self._top_n <= 0:
            raise ValueError("top_n debe ser mayor que cero.")

        columnas_requeridas = ["country", "region", "year", self._columna]
        faltantes = [col for col in columnas_requeridas if col not in self._df.columns]

        if faltantes:
            raise ValueError(f"Faltan columnas requeridas: {faltantes}")

        resultados = []

        for _, fila in self._df.iterrows():
            valor = fila[self._columna]

            if pd.notna(valor):
                resultados.append({
                    "country": fila["country"],
                    "region": fila["region"],
                    "year": int(fila["year"]),
                    "valor": float(valor)
                })

        resultados_ordenados = sorted(
            resultados,
            key=lambda x: x["valor"],
            reverse=True
        )

        return pd.DataFrame(resultados_ordenados[:self._top_n])

## 5. Subclase `MaximoFitness`

In [6]:
class MaximoFitness(AlgoritmoBase):
    """
    Algoritmo para obtener máximo de una lista numérica.
    """

    def __init__(self, valores, modo="iterativo"):
        super().__init__(f"Máximo Fitness - {modo}")

        self._valores = list(valores)
        self._modo = modo

    def _maximo_iterativo(self):
        if len(self._valores) == 0:
            return None

        maximo = self._valores[0]

        for valor in self._valores[1:]:
            if valor > maximo:
                maximo = valor

        return maximo

    def _maximo_recursivo_optimizado(self, inicio=0, fin=None):
        if fin is None:
            fin = len(self._valores)

        if inicio >= fin:
            return None

        if fin - inicio == 1:
            return self._valores[inicio]

        mitad = (inicio + fin) // 2

        max_izq = self._maximo_recursivo_optimizado(inicio, mitad)
        max_der = self._maximo_recursivo_optimizado(mitad, fin)

        if max_izq is None:
            return max_der

        if max_der is None:
            return max_izq

        return max(max_izq, max_der)

    def ejecutar(self):
        if self._modo == "iterativo":
            return self._maximo_iterativo()

        if self._modo == "recursivo":
            return self._maximo_recursivo_optimizado()

        raise ValueError("Modo inválido. Use 'iterativo' o 'recursivo'.")

## 6. Subclase `PromedioRegionalFitness`

In [7]:
class PromedioRegionalFitness(AlgoritmoBase):
    """
    Algoritmo para calcular promedio de una columna agrupada por región.
    """

    def __init__(self, dataframe, columna):
        super().__init__("Promedio Regional Fitness")

        self._df = dataframe.copy()
        self._columna = columna

    def ejecutar(self):
        columnas_requeridas = ["region", self._columna]
        faltantes = [col for col in columnas_requeridas if col not in self._df.columns]

        if faltantes:
            raise ValueError(f"Faltan columnas requeridas: {faltantes}")

        return (
            self._df
            .groupby("region")[self._columna]
            .mean()
            .sort_values(ascending=False)
        )

## 7. Clase `MedidorComplejidad`

In [8]:
class MedidorComplejidad:
    """
    Clase responsable de medir tiempo y memoria de algoritmos.
    """

    def medir_tiempo(self, funcion, *args, repeticiones=5, **kwargs):
        tiempos = []
        resultado = None

        for _ in range(repeticiones):
            inicio = time.perf_counter()
            resultado = funcion(*args, **kwargs)
            fin = time.perf_counter()

            tiempos.append(fin - inicio)

        return {
            "resultado": resultado,
            "tiempo_promedio": sum(tiempos) / len(tiempos),
            "tiempo_minimo": min(tiempos),
            "tiempo_maximo": max(tiempos),
            "repeticiones": repeticiones
        }

    def medir_memoria(self, funcion, *args, **kwargs):
        tracemalloc.start()

        resultado = funcion(*args, **kwargs)

        memoria_actual, memoria_pico = tracemalloc.get_traced_memory()

        tracemalloc.stop()

        return {
            "resultado": resultado,
            "memoria_actual_bytes": memoria_actual,
            "memoria_pico_bytes": memoria_pico,
            "memoria_pico_kb": memoria_pico / 1024,
            "memoria_pico_mb": memoria_pico / (1024 * 1024)
        }

    def medir_algoritmo(self, algoritmo, repeticiones=5):
        medicion_tiempo = self.medir_tiempo(
            algoritmo.ejecutar,
            repeticiones=repeticiones
        )

        medicion_memoria = self.medir_memoria(
            algoritmo.ejecutar
        )

        return {
            "algoritmo": algoritmo.nombre,
            "tiempo_promedio": medicion_tiempo["tiempo_promedio"],
            "tiempo_minimo": medicion_tiempo["tiempo_minimo"],
            "tiempo_maximo": medicion_tiempo["tiempo_maximo"],
            "memoria_pico_kb": medicion_memoria["memoria_pico_kb"],
            "memoria_pico_mb": medicion_memoria["memoria_pico_mb"],
            "repeticiones": repeticiones
        }

## 8. Clase `ExperimentoAlgoritmico`

Coordina ejecución polimórfica y medición.

In [9]:
class ExperimentoAlgoritmico:
    """
    Clase que coordina la ejecución polimórfica de varios algoritmos.
    """

    def __init__(self, algoritmos, medidor):
        self._algoritmos = list(algoritmos)
        self._medidor = medidor

    def ejecutar_algoritmos(self):
        resultados = {}

        for algoritmo in self._algoritmos:
            resultados[algoritmo.nombre] = algoritmo.ejecutar()

        return resultados

    def medir_algoritmos(self):
        mediciones = []

        for algoritmo in self._algoritmos:
            medicion = self._medidor.medir_algoritmo(algoritmo)
            mediciones.append(medicion)

        return pd.DataFrame(mediciones)

## 9. Ejecución POO y polimorfismo

In [10]:
dataset_fitness = DatasetFitness(df_f3)

dataset_fitness.resumen()

{'filas': 3564,
 'columnas': 8,
 'paises': 132,
 'regiones': 6,
 'anio_min': 2000,
 'anio_max': 2026}

In [11]:
valores_memberships = dataset_fitness.obtener_columna_numerica(
    "memberships_per_100k"
)

algoritmos = [
    RankingFitness(
        dataset_fitness.df,
        columna="memberships_per_100k",
        top_n=10
    ),
    MaximoFitness(
        valores_memberships,
        modo="iterativo"
    ),
    MaximoFitness(
        valores_memberships,
        modo="recursivo"
    ),
    PromedioRegionalFitness(
        dataset_fitness.df,
        columna="gym_penetration_rate"
    )
]

for algoritmo in algoritmos:
    print("Algoritmo:", algoritmo.nombre)
    resultado = algoritmo.ejecutar()

    if isinstance(resultado, pd.DataFrame) or isinstance(resultado, pd.Series):
        display(resultado)
    else:
        print(resultado)

    print("-" * 80)

Algoritmo: Ranking Fitness


,country,region,year,valor
0,United States of America,North America,2026,30221.918617
1,United States of America,North America,2025,30000.846077
2,Netherlands,Europe,2026,29194.416757
3,Norway,Europe,2023,28987.602546
4,Netherlands,Europe,2025,28980.856127
5,Netherlands,Europe,2024,28731.432516
6,Norway,Europe,2026,28597.186178
7,Norway,Europe,2025,28388.007851
8,Norway,Europe,2024,28143.691298
9,Australia,Oceania,2026,27699.213423


--------------------------------------------------------------------------------
Algoritmo: Máximo Fitness - iterativo
30221.9186167546
--------------------------------------------------------------------------------
Algoritmo: Máximo Fitness - recursivo
30221.9186167546
--------------------------------------------------------------------------------
Algoritmo: Promedio Regional Fitness


region
Europe           0.113839
Oceania          0.098022
North America    0.088760
South America    0.055014
Asia             0.050762
Africa           0.013010
Name: gym_penetration_rate, dtype: float64

--------------------------------------------------------------------------------


## 10. Medición POO

In [12]:
medidor = MedidorComplejidad()

experimento = ExperimentoAlgoritmico(
    algoritmos=algoritmos,
    medidor=medidor
)

df_mediciones_poo = experimento.medir_algoritmos()

df_mediciones_poo

,algoritmo,tiempo_promedio,tiempo_minimo,tiempo_maximo,memoria_pico_kb,memoria_pico_mb,repeticiones
0,Ranking Fitness,0.208640,0.197805,0.215045,1406.460938,1.373497,5
1,Máximo Fitness - iterativo,0.000219,0.000211,0.000242,27.882812,0.027229,5
2,Máximo Fitness - recursivo,0.002285,0.002255,0.002357,0.433594,0.000423,5
3,Promedio Regional Fitness,0.000642,0.000557,0.000876,188.487305,0.184070,5


## 11. Validaciones POO

In [13]:
assert isinstance(dataset_fitness.df, pd.DataFrame)

assert isinstance(algoritmos[0], AlgoritmoBase)
assert isinstance(algoritmos[1], AlgoritmoBase)
assert isinstance(algoritmos[2], AlgoritmoBase)

assert hasattr(algoritmos[0], "ejecutar")
assert hasattr(algoritmos[1], "ejecutar")
assert hasattr(algoritmos[2], "ejecutar")

print("Validaciones POO ejecutadas correctamente.")

try:
    algoritmo_invalido = MaximoFitness(valores_memberships, modo="modo_inexistente")
    algoritmo_invalido.ejecutar()
except ValueError as error:
    print("Error controlado correctamente:", error)

Validaciones POO ejecutadas correctamente.
Error controlado correctamente: Modo inválido. Use 'iterativo' o 'recursivo'.


## Conclusión

Este notebook cubre herencia, polimorfismo, encapsulamiento, cohesión, bajo acoplamiento y mediciones sobre objetos, fortaleciendo el cumplimiento de RA3 e indicadores ID 3.1 e ID 3.2.